# Entregable 3: entrenamiento en Colab Pro (NAGFuse y semillas adicionales del benchmark)

Este cuaderno entrena el modelo propio **NAGFuse**, las 4 semillas nuevas (8 a 11) de los 10 modelos del benchmark y las variantes de ablación, y evalúa todo en los tres conjuntos adicionales.

**Antes de empezar**
1. *Entorno de ejecución → Cambiar tipo*: **GPU L4** (mejor relación costo/tiempo) o **A100** (más rápida, gasta ~2.5 veces más unidades).
2. Google Drive con **al menos 8 GB libres**. Todo se guarda en `MyDrive/E3_TrashNet/`.
3. Para dos de los conjuntos adicionales (GD v2 y WaRP-C) se usa Kaggle. En el ícono de la llave (*Secrets*) crea `KAGGLE_USERNAME` y `KAGGLE_KEY` (de kaggle.com → Settings → API → Create New Token) y dales acceso al cuaderno.

**Qué pasa si se cae la sesión, la luz o el internet:** el entrenamiento corre en la nube de Google, no en tu computador. Si la sesión de Colab se cierra, vuelve a abrir el cuaderno y ejecuta **Entorno de ejecución → Ejecutar todas**. Las tareas terminadas se saltan y la corrida en curso se reanuda desde el último checkpoint guardado en Drive (se pierden como máximo unos 8 minutos).

## 1. Parámetros de esta sesión

In [ ]:
REF = "e3-protocolo-v1"     # tag (o rama) del repositorio con el protocolo congelado
WORKER_ID = "0"             # si abres una segunda sesión en paralelo, usa "1" en ella
HORAS = 40                  # límite de horas para lanzar tareas nuevas en esta sesión
PRUEBA_DE_HUMO = True       # la primera vez: True. Si la prueba pasa, cámbialo a False y vuelve a ejecutar todo
ACCUM = 1                   # acumulación de gradiente si falta memoria de GPU (el lote efectivo sigue en 32)
REPO_URL = "https://github.com/Anyita-12/deep-learning-colaborativo.git"

## 2. Drive, repositorio y dependencias

In [ ]:
import os, subprocess, sys, json, signal
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

REPO = Path("/content/deep-learning-colaborativo")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", f"+refs/tags/{REF}:refs/tags/{REF}"], check=False)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", REF], check=False)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-f", "FETCH_HEAD"], check=True)
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Commit en uso:", commit)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "entregable3" / "requirements-train.txt")], check=True)
import torch
print(torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO HAY GPU")
assert torch.cuda.is_available(), "Activa una GPU en Entorno de ejecución → Cambiar tipo de entorno"

E3 = REPO / "entregable3"
os.chdir(E3)
os.environ["PYTHONPATH"] = str(E3)
os.environ["TRASHNET_ACCUM"] = str(ACCUM)
os.environ["E3_WORK"] = "/content/e3_work"
os.environ["E3_DRIVE"] = "/content/drive/MyDrive/E3_TrashNet"
Path(os.environ["E3_DRIVE"]).mkdir(parents=True, exist_ok=True)
(Path(os.environ["E3_DRIVE"]) / "commit_en_uso.txt").write_text(f"{REF} {commit}\n")

## 3. Credenciales de Kaggle (para GD v2 y WaRP-C)

In [ ]:
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Credenciales de Kaggle cargadas desde Secrets.")
except Exception as e:
    kj = Path(os.environ["E3_DRIVE"]) / "kaggle.json"
    if kj.exists():
        k = json.loads(kj.read_text()); os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = k["username"], k["key"]
        print("Credenciales de Kaggle cargadas desde Drive/E3_TrashNet/kaggle.json.")
    else:
        print("Sin credenciales de Kaggle: GD v2 y WaRP-C quedarán pendientes (el entrenamiento sigue).", e)

## 4. Pruebas unitarias del modelo propio (1 a 2 minutos)

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=E3)
assert r.returncode == 0, "Las pruebas fallaron: no lances el entrenamiento."

## 5. Ejecución

Esta celda lanza el orquestador en un proceso aparte y muestra su registro. Si `PRUEBA_DE_HUMO = True`, solo corre la prueba de humo (unos 15 a 25 minutos, en carpetas `_smoke`, sin tocar los resultados reales).

Para detener: usa el botón de parar de la celda (o la celda 6). El progreso queda guardado.

In [ ]:
def lanzar(*args):
    p = subprocess.Popen([sys.executable, "-m", "e3.orchestrator", *args], cwd=E3, env=dict(os.environ),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    (Path("/content") / "orquestador.pid").write_text(str(p.pid))
    try:
        for line in p.stdout:
            print(line, end="")
    except KeyboardInterrupt:
        os.killpg(p.pid, signal.SIGTERM)
        print("\nDetenido. El progreso hasta el último checkpoint está en Drive.")
    return p.wait()

if PRUEBA_DE_HUMO:
    rc = lanzar("smoke")
    print("Código de salida:", rc, "→", "PRUEBA SUPERADA: cambia PRUEBA_DE_HUMO a False" if rc == 0 else "REVISA LOS ERRORES")
else:
    assert lanzar("setup") == 0
    assert lanzar("prepare", "--worker", WORKER_ID) == 0
    lanzar("run", "--worker", WORKER_ID, "--hours", str(HORAS))

## 6. Detener (solo si hace falta)

In [ ]:
pid_file = Path("/content/orquestador.pid")
if pid_file.exists():
    try:
        os.killpg(int(pid_file.read_text()), signal.SIGTERM); print("Orquestador detenido.")
    except ProcessLookupError:
        print("No había un orquestador en ejecución.")

## 7. Estado (se puede ejecutar en cualquier momento, también desde otra sesión)

In [ ]:
subprocess.run([sys.executable, "-m", "e3.orchestrator", "status"], cwd=E3, env=dict(os.environ))

## 8. Evaluación pendiente en conjuntos adicionales (si alguno falló al descargarse)

In [ ]:
# lanzar("extra-pass", "--worker", WORKER_ID)